# SAR-ATR Colab 실행 템플릿
**사용법**: Cell 1 → Cell 2 순서로 실행. GPU 런타임 권장.

| 변수 | 설명 |
|---|---|
| `REPO_URL` | 팀 GitHub 레포 URL |
| `DRIVE_ROOT` | Google Drive 공유 폴더 경로 |
| `EXP` | 실행할 실험 (`a`, `b`, `c`, `d`) |

In [ ]:
# ── Cell 1: 환경 설정 (모든 노트북 공통) ──────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/SAR_ATR_Project'
REPO_URL   = 'https://github.com/<your-org>/sar-atr.git'  # TODO: 팀 레포 URL로 교체

import os, sys
if not os.path.exists('/content/repo'):
    !git clone {REPO_URL} /content/repo
%cd /content/repo
!git pull
!pip install -r requirements.txt -q
sys.path.insert(0, '/content/repo')

# Drive symlinks: data/results는 Drive에서 읽고 씀
os.makedirs(f'{DRIVE_ROOT}/data', exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/results', exist_ok=True)
!ln -sf {DRIVE_ROOT}/data    /content/repo/data
!ln -sf {DRIVE_ROOT}/results /content/repo/results

import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# ── Cell 2: 빠른 파이프라인 검증 (mock data) ─────────────────────────
from core.mock_data import MockSARDataset
from core.models import get_model
from core.train import train_model
from core.evaluate import evaluate
from core.interfaces import TrainConfig

train_ds = MockSARDataset(n=200, num_classes=3, seed=0)
val_ds   = MockSARDataset(n=60,  num_classes=3, seed=1)

for model_name in ['smpl', 'resnet18']:
    model  = get_model(model_name, num_classes=3)
    config = TrainConfig(model_name=model_name, num_classes=3, epochs=5, seed=0)
    model, result = train_model(model, train_ds, val_ds, config)
    print(f'[{model_name}] mock val acc = {result.accuracy*100:.1f}%')

In [ ]:
# ── Cell 3: Exp A — 클러터 전이 (Table 4) ────────────────────────────
# 실 데이터: data/clutter_gengzhe/ 필요 (없으면 mock으로 대체됨)
import sys
sys.argv = ['']  # argparse 충돌 방지

from experiments.exp_a_clutter_transfer import run_all
results_a = run_all(epochs=60)

In [ ]:
# ── Cell 4: Exp C — 밝기 보정 + Optuna (Figure 1) ────────────────────
# 실 데이터: data/mstar/mixed_targets/ 필요
from experiments.exp_c_contrast_optuna import run as run_c
results_c = run_c(model_name='resnet18', n_optuna_trials=20, epochs_full=60, epochs_trial=10)

In [ ]:
# ── Cell 5: Exp D — OOD 탐지 ─────────────────────────────────────────
from experiments.exp_d_ood import run as run_d
results_d = run_d(model_name='smpl', j_list=[1, 2, 3], epochs=60)

In [ ]:
# ── Cell 6: 결과 커밋 (세션 종료 전 실행) ────────────────────────────
%cd /content/repo
!git config user.email 'colab@sar-atr'
!git config user.name  'Colab Runner'
!git add results/*/metrics.json results/*/*.png results/*/*.pth
!git commit -m 'exp: Colab 실행 결과 업데이트' || echo 'nothing to commit'
!git push